In [1]:
import numpy as np
import tensorflow as tf
import pickle
import functools

In [2]:
from tensorflow.compat.v1 import ConfigProto
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1"
config=tf.compat.v1.ConfigProto()
config.gpu_options.per_process_gpu_memory_fraction = 0.8
config.gpu_options.allow_growth=True
sess=tf.compat.v1.Session(config=config) 

In [3]:
from network import build_mlp_mixer_utkface_age
from train import WarmUpCosine, AdamW, LastNSaver, RSquared, cifar_pad_crop_flip, mixup_batch, randaugment_mild, make_test_ds
from noise import noise_a, noise_p 

In [4]:
num_classes = 3
initial_lr = 5e-4
weight_decay = 1e-5
epochs = 200
warmup_epochs = 5
batch_size = 128
image_size = 32

In [5]:
SAVE_PATH = "utkface_12k_split.npz" 
def load_dataset_if_exists(path=SAVE_PATH):
    """
    Load if file exists, otherwise return None.
    """
    if os.path.exists(path):
        data = np.load(path)
        print("✔ Cached data detected, loading directly")
        return (data["x_train"], data["y_train"],
                data["x_test"], data["y_test"])
    return None

In [6]:
x_train, y_train, x_test, y_test = load_dataset_if_exists()
y_train = tf.cast(y_train, dtype=tf.float32)
y_test = tf.cast(y_test, dtype=tf.float32)

✔ 检测到缓存数据，已直接加载


In [7]:
model = build_mlp_mixer_utkface_age(num_classes=1)

In [8]:
total_steps = epochs * (x_train.shape[0] // batch_size)
warmup_steps = warmup_epochs * (x_train.shape[0] // batch_size)
lr_schedule = WarmUpCosine(initial_lr, total_steps, warmup_steps)
optimizer = AdamW(
    learning_rate=lr_schedule,
    weight_decay=weight_decay
)
model.compile(
    optimizer=optimizer,
    loss='mse',
    metrics=[RSquared()]
)

In [9]:
saver = LastNSaver(n=20)

In [10]:
gaussian_noise_batch = functools.partial(noise_a, a=0.2)
salt_pepper_noise_batch = functools.partial(noise_p, n=64)

In [11]:
def random_noise_batch(x, p_gaussian=0.5):
    x_g = gaussian_noise_batch(x)
    x_sp = salt_pepper_noise_batch(x)

    b = tf.shape(x)[0]
    use_g = tf.random.uniform([b, 1, 1, 1]) < p_gaussian
    use_g = tf.cast(use_g, x.dtype)

    return use_g * x_g + (1.0 - use_g) * x_sp


def clean_noisy_batch(x, y):
    x_noisy = random_noise_batch(x)

    x = tf.concat([x, x_noisy], axis=0)
    y = tf.concat([y, y], axis=0)
    return x, y

In [12]:
def make_train_ds(x_train, y_train_onehot, batch_size=128,
                  image_size=32, pad=4,
                  mixup_alpha=0.0,
                  strong_aug=False,
                  use_noise=True):

    ds = tf.data.Dataset.from_tensor_slices((x_train, y_train_onehot))
    ds = ds.shuffle(min(len(x_train), 20000), reshuffle_each_iteration=True)

    def to_float(x, y):
        x = tf.cast(x, tf.float32)
        y = tf.cast(y, tf.float32)
        return x, y

    ds = ds.map(to_float, num_parallel_calls=AUTOTUNE)

    ds = ds.map(
        lambda x, y: cifar_pad_crop_flip(x, y, image_size=image_size, pad=pad),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.map(
        lambda x, y: randaugment_mild(x, y, strong_aug=strong_aug),
        num_parallel_calls=AUTOTUNE
    )

    ds = ds.batch(batch_size, drop_remainder=True)

    if use_noise:
        ds = ds.map(clean_noisy_batch, num_parallel_calls=AUTOTUNE)

    if mixup_alpha and mixup_alpha > 0:
        ds = ds.map(
            lambda xb, yb: mixup_batch(xb, yb, alpha=mixup_alpha),
            num_parallel_calls=AUTOTUNE
        )

    ds = ds.prefetch(AUTOTUNE)
    return ds

In [13]:
AUTOTUNE = tf.data.AUTOTUNE
train_ds = make_train_ds(
    x_train, y_train,
    batch_size=batch_size,
    image_size=32,
    pad=4,
    mixup_alpha=0.05,
    strong_aug=False,
    use_noise=True
)
test_ds = make_test_ds(x_test, y_test, batch_size=batch_size)

In [14]:
model.fit(
    train_ds,
    epochs=epochs,
    validation_data=test_ds,
    verbose=2,
    callbacks=[saver]
)

Epoch 1/200
156/156 - 21s - loss: 0.0362 - r_squared: -2.0635e-01 - val_loss: 0.0417 - val_r_squared: -2.7727e-01 - 21s/epoch - 137ms/step
Epoch 2/200
156/156 - 11s - loss: 0.0256 - r_squared: 0.1499 - val_loss: 0.0288 - val_r_squared: 0.1196 - 11s/epoch - 68ms/step
Epoch 3/200
156/156 - 10s - loss: 0.0226 - r_squared: 0.2546 - val_loss: 0.0230 - val_r_squared: 0.2963 - 10s/epoch - 67ms/step
Epoch 4/200
156/156 - 10s - loss: 0.0210 - r_squared: 0.3025 - val_loss: 0.0200 - val_r_squared: 0.3884 - 10s/epoch - 67ms/step
Epoch 5/200
156/156 - 10s - loss: 0.0196 - r_squared: 0.3504 - val_loss: 0.0179 - val_r_squared: 0.4507 - 10s/epoch - 67ms/step
Epoch 6/200
156/156 - 10s - loss: 0.0188 - r_squared: 0.3753 - val_loss: 0.0176 - val_r_squared: 0.4621 - 10s/epoch - 67ms/step
Epoch 7/200
156/156 - 10s - loss: 0.0179 - r_squared: 0.4094 - val_loss: 0.0169 - val_r_squared: 0.4843 - 10s/epoch - 67ms/step
Epoch 8/200
156/156 - 10s - loss: 0.0171 - r_squared: 0.4314 - val_loss: 0.0158 - val_r_squar

In [15]:
model.save("Mix_8.h5")